# Datasets: Type-Safe Structured APIs
* **Datasets** are Spark's **type-safe structured API** for **Java and Scala**. They are **not available in Python or R** because those languages are dynamically typed.
* Unlike **DataFrames**, which store generic **Row** objects, **Datasets** store **strongly typed objects** (such as `Person`), allowing compile-time type checking.
* The Dataset API prevents treating data as the wrong type, making it safer and more reliable for **large applications** with multiple developers.
* Datasets are parameterized by a type, such as `Dataset<Person>`, ensuring every record matches that class.
* Spark supports **JavaBeans** in Java and **case classes** in Scala because it can automatically infer their schema.
* Developers can **switch between Datasets and DataFrames** as needed:

  * Use **Datasets** for type-safe operations with custom `map` and `filter` functions.
  * Convert back to **DataFrames** to take advantage of Spark's rich SQL and built-in functions.
* This flexibility lets developers combine the **safety of typed programming** with the **power and convenience of Spark SQL**.


In [0]:
%scala
case class Flight(DEST_COUNTRY_NAME: String, ORIGIN_COUNTRY_NAME: String, count: BigInt)
val flightsDF = spark.read.parquet("/Workspace/Users/jcrmendes97@gmail.com/Spark-The-Definitive-Guide/data/flight-data/parquet/2010-summary.parquet")
val flights = flightsDF.as[Flight]

One final advantage is that when you call collect or take on a Dataset, it will collect objects of
the proper type in your Dataset, not DataFrame Rows. This makes it easy to get type safety and
securely perform manipulation in a distributed and a local manner without code changes:

In [0]:
%scala
flights
.filter(flight_row => flight_row.ORIGIN_COUNTRY_NAME !="Canada")
.map(flight_row => flight_row)
.take(5)

# Structured Streaming

* **Structured Streaming** is Spark's **high-level API for stream processing**, introduced as production-ready in **Spark 2.2**.
* It allows you to use the **same DataFrame and Dataset operations** for both **batch** and **streaming** data, requiring **minimal code changes** when converting a batch application to a streaming one.
* Streaming in Spark means **near real-time processing**, not hard real-time. Data is processed continuously as it arrives, typically with **latencies of milliseconds to seconds**, making it suitable for dashboards, monitoring, fraud detection, and other live analytics.
* It supports **incremental processing**, where only newly arriving data is processed instead of reprocessing the entire dataset, reducing latency and improving efficiency.
* Structured Streaming treats incoming data as an **unbounded table** that continuously grows over time. Developers write the same DataFrame/Dataset transformations as they would for batch processing, and Spark automatically applies them to new records as they arrive.
* This unified programming model allows developers to **prototype applications using batch processing** and then **easily adapt them to handle continuous data streams**, combining ease of development with scalable, near real-time analytics.


In [0]:
staticDataFrame = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load("/Workspace/Users/jcrmendes97@gmail.com/Spark-The-Definitive-Guide/data/retail-data/by-day/*.csv")
staticDataFrame.createOrReplaceTempView("retail_data")
staticSchema = staticDataFrame.schema


import pyspark.sql.functions as F
spark.conf.set("spark.sql.shuffle.partitions","200")
staticDataFrame.selectExpr(
    "CustomerId",
    "(UnitPrice * Quantity) as total_cost",
    "InvoiceDate"
).groupBy(F.col("CustomerID"), F.window(F.col("InvoiceDate"), "1 day")).sum("total_cost").show(5)

# With partitions = 5
# L13 - 9s 322 ms
# L2 - 21s 57 ms

# With default partitons = 200
# L13 - 7s 531 ms
# L2 - 17s 640 ms


+----------+--------------------+-----------------+
|CustomerID|              window|  sum(total_cost)|
+----------+--------------------+-----------------+
|   16057.0|{2011-12-05 00:00...|            -37.6|
|   14126.0|{2011-11-29 00:00...|643.6300000000001|
|   13500.0|{2011-11-16 00:00...|497.9700000000001|
|   17160.0|{2011-11-08 00:00...|516.8499999999999|
|   15608.0|{2011-11-11 00:00...|            122.4|
+----------+--------------------+-----------------+
only showing top 5 rows


# Why is spark.sql.shuffle.partitions important?
* `spark.sql.shuffle.partitions` controls **the number of partitions Spark creates after a shuffle operation**.
* A **shuffle** redistributes data across partitions so related records are grouped together. Common operations that trigger a shuffle include:

  * `groupBy()`
  * `join()`
  * `distinct()`
  * `orderBy()`
  * `repartition()`
* The default value is **200**, which is appropriate for **large Spark clusters** with many executors and CPU cores.
* When running Spark in **local mode** (e.g., on a laptop), 200 partitions often create **too many small tasks**, increasing scheduling overhead and reducing performance.
* Reducing the number of shuffle partitions (e.g., to **5**) better matches the limited resources of a local machine, resulting in **fewer tasks, larger partitions, and faster execution** for small datasets.
* **Rule of thumb:**

  * **Local mode / small datasets:** Use fewer shuffle partitions (e.g., 4–10).
  * **Large clusters:** Keep the default or tune it based on the cluster size and workload.


In [0]:
# How to list the partitions available
spark.conf.get("spark.sql.shuffle.partitions")

'200'

## Streaming example

In [0]:
streamingDataFrame = (
    spark
    .readStream
    .schema(staticSchema)
    .option("maxFilesPerTrigger", 1)
    .format("csv")
    .option("header", "true")
    .load("/Workspace/Users/jcrmendes97@gmail.com/Spark-The-Definitive-Guide/data/retail-data/by-day/*.csv")
)

# Now we can see whether our DataFrame is streaming
streamingDataFrame.isStreaming

purchaseByCustomerPerHour = (
    streamingDataFrame
    .selectExpr(
        "CustomerId",
        "(UnitPrice * Quantity) as total_cost",
        "InvoiceDate"
    )
    .groupBy(
        F.col("CustomerID"), F.window(F.col("InvoiceDate"), "1 day")
    )
    .sum("total_cost")
)

# Action for the Structured Streaming
(
    purchaseByCustomerPerHour
    .writeStream
    .format("memory")
    .queryName("customer_purchases")
    .outputMode("complete")
    .start()
)

# After starting the stream, we can look at the data
spark.sql("""
    SELECT *
    FROM customer_purchases
    ORDER BY `sum(total_cost)` DESC
""").show(5)



---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-5878687511610215>, line 34
     14 purchaseByCustomerPerHour = (
     15     streamingDataFrame
     16     .selectExpr(
   (...)
     24     .sum("total_cost")
     25 )
     27 # Action for the Structured Streaming
     28 (
     29     purchaseByCustomerPerHour
     30     .writeStream
     31     .format("memory")
     32     .queryName("customer_purchases")
     33     .outputMode("complete")
---> 34     .start()
     35 )
     37 # After starting the stream, we can look at the data
     38 spark.sql("""
     39     SELECT *
     40     FROM customer_purchases
     41     ORDER BY `sum(total_cost)` DESC
     42 """).show(5)

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/streaming/readwriter.py:723, in DataStreamWriter.start(self, path, format, outputMode, partitionBy, queryName, **options)
 

## Machine Learning and Advanced Analytics

**What is K-Means?**
**K-Means** is an algorithm that groups similar data into **K clusters**.

**How it works:**

1. Choose **K** (the number of groups).
2. Randomly place **K centroids** (center points).
3. Assign each data point to the **closest centroid**.
4. Move each centroid to the **average (mean)** of the points assigned to it.
5. Repeat steps 3 and 4 until the centroids stop moving.

**Example:**
Imagine you have 100 customers and want to divide them into **3 groups (K = 3)** based on their age and annual spending. K-Means starts with 3 random centroids, assigns each customer to the nearest centroid, moves the centroids to the average location of each group, and repeats the process. Eventually, you end up with three clusters, such as:

* Young, low spenders
* Middle-aged, medium spenders
* Older, high spenders

**In one sentence:** K-Means repeatedly assigns points to the nearest center and updates the centers until the groups no longer change.


In [0]:
# ============================================================
# 1. PREPARE THE DATA
# ============================================================

# Replace missing values (NULL) with 0 because Machine Learning
# algorithms cannot work with missing values.
#
# Example:
# Quantity = NULL  -->  Quantity = 0
#
# Also create a new column "day_of_week" by extracting the weekday
# from InvoiceDate.
#
# Example:
# InvoiceDate = 2011-06-01  -->  day_of_week = "Wednesday"
#
# Finally, reduce the number of partitions to 5 to improve performance
# for this dataset size.
preppedDataFrame = (
    staticDataFrame
    .na.fill(0)
    .withColumn("day_of_week", F.date_format(F.col("InvoiceDate"), "EEEE"))
    .coalesce(5)
)


# ============================================================
# 2. SPLIT DATA INTO TRAINING AND TEST DATA
# ============================================================

# Training data:
# Transactions before July 2011.
# This data is used to learn the clusters.
trainDataFrame = preppedDataFrame.where("InvoiceDate < '2011-07-01'")


# Testing data:
# Transactions from July 2011 onwards.
# This data is used to check if the model works on unseen data.
testDataFrame = preppedDataFrame.where("InvoiceDate >= '2011-07-01'")


# Check how many records exist in each dataset.
print(trainDataFrame.count())
print(testDataFrame.count())


# ============================================================
# 3. CONVERT CATEGORICAL DATA INTO NUMBERS
# ============================================================

from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml import Pipeline


# StringIndexer converts text categories into numbers.
#
# Example:
# Monday    -> 0
# Tuesday   -> 1
# Friday    -> 2
#
# Machine Learning models cannot directly understand text.
indexer = (
    StringIndexer()
    .setInputCol("day_of_week")
    .setOutputCol("day_of_week_index")
)


# OneHotEncoder converts the category number into a vector.
#
# Why?
# Because K-Means calculates distances.
# If we keep:
# Monday = 0
# Tuesday = 1
# Friday = 2
#
# The model may think Friday is "bigger" than Monday.
# One-hot encoding avoids this.
#
# Example:
# Monday  -> [1,0,0]
# Tuesday -> [0,1,0]
# Friday  -> [0,0,1]
encoder = (
    OneHotEncoder()
    .setInputCol("day_of_week_index")
    .setOutputCol("day_of_week_encoded")
)


# Combine all features into a single vector.
#
# K-Means requires input in the form:
# features = [feature1, feature2, feature3, ...]
#
# Example:
# UnitPrice = 5
# Quantity = 3
# Monday = [1,0,0]
#
# Becomes:
# features = [5,3,1,0,0]
vectorAssembler = (
    VectorAssembler()
    .setInputCols([
        "UnitPrice",
        "Quantity",
        "day_of_week_encoded"
    ])
    .setOutputCol("features")
)


# ============================================================
# 4. CREATE THE TRANSFORMATION PIPELINE
# ============================================================

# A Pipeline executes all preprocessing steps in order:
#
# Raw data
#     |
#     ↓
# Convert weekday text into numbers
#     |
#     ↓
# One-hot encode weekday
#     |
#     ↓
# Create final feature vector
#
transformationPipeline = Pipeline().setStages([
    indexer,
    encoder,
    vectorAssembler
])


# Learn the transformation rules using training data.
#
# Example:
# The pipeline learns:
# Monday -> 0
# Tuesday -> 1
# Wednesday -> 2
#
fittedPipeline = transformationPipeline.fit(trainDataFrame)


# Apply the transformations to training data.
#
# The result contains the "features" column required by K-Means.
transformedTraining = fittedPipeline.transform(trainDataFrame)


# ============================================================
# 5. TRAIN THE K-MEANS MODEL
# ============================================================

from pyspark.ml.clustering import KMeans


# Create a K-Means algorithm.
#
# K = 20 means:
# "Find 20 different groups (clusters) in the data."
#
# Seed = 1 ensures the random initialization is reproducible.
# Running the model multiple times will start from the same point.
kmeans = KMeans().setK(20).setSeed(1)


# Train the model.
#
# K-Means will:
# 1. Randomly create 20 centroids.
# 2. Assign each transaction to the closest centroid.
# 3. Move centroids to the average position of their points.
# 4. Repeat until the clusters stop changing.
kmModel = kmeans.fit(transformedTraining)


# ============================================================
# 6. TEST THE MODEL
# ============================================================

# Apply the same preprocessing pipeline to test data.
#
# Important:
# Test data must have the exact same transformations
# as training data.
transformedTest = fittedPipeline.transform(testDataFrame)


# Calculate the clustering cost.
#
# It measures how far points are from their assigned centroid.
#
# Lower cost:
# - Points are closer to their clusters.
# - Clusters are more compact.
#
# Higher cost:
# - Points are more spread out.
#
# Example:
# Distance values:
# 1, 2, 4
#
# Cost:
# 1² + 2² + 4² = 21
kmModel.computeCost(transformedTest)

245903
296006


---------------------------------------------------------------------------
SparkException                            Traceback (most recent call last)
File <command-5878687511610218>, line 24
     22 from pyspark.ml.clustering import KMeans
     23 kmeans = KMeans().setK(20).setSeed(1)
---> 24 kmModel = kmeans.fit(transformedTraining)
     26 transformedTest = fittedPipeline.transform(testDataFrame)
     27 kmModel.computeCost(transformedTest)

File /databricks/python_shell/lib/dbruntime/MLWorkloadsInstrumentation/_pyspark.py:30, in _create_patch_function.<locals>.patched_method(self, *args, **kwargs)
     28 call_succeeded = False
     29 try:
---> 30     result = original_method(self, *args, **kwargs)
     31     call_succeeded = True
     32     return result

File /databricks/python/lib/python3.12/site-packages/pyspark/ml/base.py:203, in Estimator.fit(self, dataset, params)
    201         return self.copy(params)._fit(dataset)
    202     else:
--> 203         return self._fit(da

In [0]:
# Creating an RDD
from pyspark.sql import Row
spark.sparkContext.parallelize([Row(1), Row(2), Row(3)]).toDF()

---------------------------------------------------------------------------
PySparkAttributeError                     Traceback (most recent call last)
File <command-6514582727699464>, line 2
      1 from pyspark.sql import Row
----> 2 spark.sparkContext.parallelize([Row(1), Row(2), Row(3)]).toDF()

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/session.py:1130, in SparkSession.__getattr__(self, name)
   1128 def __getattr__(self, name: str) -> Any:
   1129     if name in ["_jsc", "_jconf", "_jvm", "_jsparkSession", "sparkContext", "newSession"]:
-> 1130         raise PySparkAttributeError(
   1131             errorClass="JVM_ATTRIBUTE_NOT_SUPPORTED", messageParameters={"attr_name": name}
   1132         )
   1133     return object.__getattribute__(self, name)

PySparkAttributeError: [JVM_ATTRIBUTE_NOT_SUPPORTED] SparkContext is not supported on serverless compute. If you require direct access to the SparkContext, switch to Dedicated access mode. For more deta